In [12]:
import numpy as np
import pandas as pd

# === Paramètres ===
Te = 1.0
T = 200
sigma_theta = 0.05
sigma_r = 5
sigma_Q = 1.0

# Matrices du modèle
F = np.array([[1, Te, 0, 0], [0, 1, 0, 0],
              [0, 0, 1, Te], [0, 0, 0, 1]])
Q = sigma_Q**2 * np.array([[Te**3/3, Te**2/2, 0, 0],
                           [Te**2/2, Te, 0, 0],
                           [0, 0, Te**3/3, Te**2/2],
                           [0, 0, Te**2/2, Te]])

def h(x):
    px, _, py, _ = x
    return np.array([np.arctan2(py, px), np.sqrt(px**2 + py**2)])

def jacobian_h(x):
    px, _, py, _ = x
    rho2 = max(px**2 + py**2, 1e-6)
    rho = np.sqrt(rho2)
    return np.array([[-py / rho2, 0, px / rho2, 0],
                     [px / rho, 0, py / rho, 0]])

def kl_divergence_gaussians(mu0, P0, mu1, P1):
    n = mu0.shape[0]
    inv_P1 = np.linalg.inv(P1)
    delta = mu1 - mu0
    term1 = np.trace(inv_P1 @ P0)
    term2 = delta.T @ inv_P1 @ delta
    sign0, logdet0 = np.linalg.slogdet(P0)
    sign1, logdet1 = np.linalg.slogdet(P1)
    if sign0 <= 0 or sign1 <= 0:
        return np.nan
    term3 = logdet1 - logdet0
    return 0.5 * (term1 + term2 - n + term3)

# === Initialisation ===
x_init = np.array([3, 40, -4, 20])
x_true = np.zeros((4, T))
x_est = np.zeros((4, T))
x_true[:, 0] = x_init
x_est[:, 0] = x_init
P = np.eye(4)
sensor_choices = []

# Générer la trajectoire réelle
for k in range(1, T):
    x_true[:, k] = F @ x_true[:, k-1] + np.random.multivariate_normal(np.zeros(4), Q)

# Définir les matrices de covariance des capteurs
R_list = [
    np.diag([sigma_theta**2 + np.random.uniform(0, 0.002),
             sigma_r**2 + np.random.uniform(0, 4)])
    for _ in range(3)
]

# Générer les observations bruitées
y_all = np.zeros((3, 2, T))
for i in range(3):
    for k in range(T):
        y_all[i, :, k] = h(x_true[:, k]) + np.random.multivariate_normal(np.zeros(2), R_list[i])

# === Filtrage EKF avec sélection par divergence KL ===
for k in range(1, T):
    x_pred = F @ x_est[:, k-1]
    P_pred = F @ P @ F.T + Q

    best_kl = -np.inf
    best_i = 0
    best_xk = None
    best_Pk = None

    for i in range(3):
        Hk = jacobian_h(x_pred)
        R_i = R_list[i]
        y_obs = y_all[i, :, k]
        S = Hk @ P_pred @ Hk.T + R_i
        S += 1e-6 * np.eye(2)  # pour stabilité numérique
        K = P_pred @ Hk.T @ np.linalg.inv(S)
        x_upd = x_pred + K @ (y_obs - h(x_pred))
        P_upd = (np.eye(4) - K @ Hk) @ P_pred
        dkl = kl_divergence_gaussians(x_upd, P_upd, x_pred, P_pred)

        if dkl > best_kl:
            best_kl = dkl
            best_i = i
            best_xk = x_upd
            best_Pk = P_upd

    sensor_choices.append(best_i)
    x_est[:, k] = best_xk
    P = best_Pk

# === Créer le tableau pandas des résultats ===
df_selection = pd.DataFrame({
    "Temps": np.arange(1, T),
    "Capteur choisi (KL)": [i + 1 for i in sensor_choices]
})

# Affichage
print(df_selection)


     Temps  Capteur choisi (KL)
0        1                    2
1        2                    1
2        3                    2
3        4                    3
4        5                    3
..     ...                  ...
194    195                    2
195    196                    2
196    197                    3
197    198                    1
198    199                    1

[199 rows x 2 columns]


In [13]:
# Comptage du nombre de sélections par capteur
compte_capteurs = df_selection["Capteur choisi (KL)"].value_counts().sort_index()

# Affichage clair
print("\nNombre de fois où chaque capteur a été choisi :")
for capteur, count in compte_capteurs.items():
    print(f"Capteur {capteur} : {count} fois")



Nombre de fois où chaque capteur a été choisi :
Capteur 1 : 74 fois
Capteur 2 : 62 fois
Capteur 3 : 63 fois


In [14]:
import plotly.graph_objects as go

# Initialisation des données cumulées
cumulative_counts = np.zeros((T, 3))
for k in range(T - 1):
    cumulative_counts[k + 1] = cumulative_counts[k]
    cumulative_counts[k + 1, sensor_choices[k]] += 1

# Création des frames pour l'animation
frames = []
for k in range(1, T):
    frame = go.Frame(
        data=[
            go.Scatter(
                x=[1, 2, 3],
                y=[1, 1, 1],
                mode="markers+text",
                marker=dict(size=10 + cumulative_counts[k] * 2, color=["red", "green", "blue"]),
                text=[f"{int(c)}" for c in cumulative_counts[k]],
                textposition="top center"
            )
        ],
        name=str(k)
    )
    frames.append(frame)

# Création de la figure principale
fig_bubble = go.Figure(
    data=[
        go.Scatter(
            x=[1, 2, 3],
            y=[1, 1, 1],
            mode="markers+text",
            marker=dict(size=10 + cumulative_counts[0] * 2, color=["red", "green", "blue"]),
            text=["0", "0", "0"],
            textposition="top center"
        )
    ],
    layout=go.Layout(
        title="Évolution du nombre de sélections par capteur (KL)",
        xaxis=dict(title="Capteurs", tickvals=[1, 2, 3], ticktext=["Capteur 1", "Capteur 2", "Capteur 3"], range=[0.5, 3.5]),
        yaxis=dict(showticklabels=False, range=[0.5, 1.5]),
        updatemenus=[dict(type="buttons", showactive=False,
                          buttons=[dict(label="Play", method="animate", args=[None, {"frame": {"duration": 200, "redraw": True}, "fromcurrent": True}])])]
    ),
    frames=frames
)

fig_bubble.show()
